In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import os
import glob
from tqdm import tqdm
from collections import Counter

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
image_folder = "../data/raw/open_set"

image_paths = []
labels = []

class_mapping = {
    "benign": 0,
    "malignant": 1,
    "non_histopathology": 2
}

for class_name in class_mapping:
    class_dir = os.path.join(image_folder, class_name)
    
    for file in os.listdir(class_dir):
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            image_paths.append(os.path.join(class_dir, file))
            labels.append(class_mapping[class_name])

print("Total images:", len(image_paths))
print("Class counts:", {k: labels.count(v) for k,v in class_mapping.items()})

Total images: 11031
Class counts: {'benign': 2480, 'malignant': 5429, 'non_histopathology': 3122}


In [4]:
class OpenSetDataset(Dataset):
    def __init__(self, image_paths, labels, transform):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label

In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
dataset = OpenSetDataset(image_paths, labels, transform)

dataloader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True
)

print("Total batches:", len(dataloader))

Total batches: 173


In [17]:
model = models.resnet18(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True

model.fc = nn.Linear(model.fc.in_features, 3)

model = model.to(device)

print("Model ready.")

Model ready.


In [18]:
class_counts = Counter(labels)
print("Class counts:", class_counts)

total_samples = sum(class_counts.values())

weights = []
for i in range(3):   
    weights.append(total_samples / class_counts[i])

class_weights = torch.tensor(weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

Class counts: Counter({1: 5429, 2: 3122, 0: 2480})


In [19]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels_batch in tqdm(dataloader):
        
        images = images.to(device)
        labels_batch = labels_batch.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels_batch)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels_batch).sum().item()
        total += labels_batch.size(0)
    
    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f} | Accuracy: {accuracy:.2f}%")

100%|████████████████████████████████████████████████████████████████████████████████| 173/173 [12:32<00:00,  4.35s/it]


Epoch 1 | Loss: 27.4028 | Accuracy: 93.30%


100%|████████████████████████████████████████████████████████████████████████████████| 173/173 [10:18<00:00,  3.57s/it]


Epoch 2 | Loss: 11.5170 | Accuracy: 97.22%


100%|████████████████████████████████████████████████████████████████████████████████| 173/173 [39:41<00:00, 13.76s/it]


...

Epoch 8 | Loss: 3.0351 | Accuracy: 99.30%


100%|████████████████████████████████████████████████████████████████████████████████| 173/173 [11:07<00:00,  3.86s/it]


Epoch 9 | Loss: 2.4668 | Accuracy: 99.41%


100%|████████████████████████████████████████████████████████████████████████████████| 173/173 [11:50<00:00,  4.11s/it]

Epoch 10 | Loss: 2.2656 | Accuracy: 99.62%


In [20]:
os.makedirs("../models", exist_ok=True)

torch.save(model.state_dict(), "../models/cnn_open_set_validator.pt")

print("3-class open-set validator saved successfully.")

3-class open-set validator saved successfully.
